In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [28]:
macro = pd.read_csv(
    "../data/processed/macro_monthly.csv",
    parse_dates=["date"]
)

macro.head()

,date,unemployment,yield_curve,fed_funds_rate,cpi,industrial_production,housing_starts,initial_claims,consumer_sentiment,recession
0,1990-01-01,5.4,0.121429,8.23,127.5,61.7290,1551.0,361000.0,93.0,0
1,1990-02-01,5.3,0.102632,8.24,128.0,62.2896,1437.0,358250.0,89.5,0
2,1990-03-01,5.2,-0.038182,8.28,128.6,62.5999,1289.0,345200.0,91.3,0
3,1990-04-01,5.4,0.061500,8.26,128.9,62.4359,1248.0,361750.0,93.9,0
4,1990-05-01,5.4,0.115909,8.18,129.1,62.6258,1212.0,355250.0,90.6,0


In [29]:
macro["inflation_yoy"] = (
    macro["cpi"].pct_change(12) * 100
)

In [30]:
macro["industrial_growth_yoy"] = (
    macro["industrial_production"].pct_change(12) * 100
)

In [31]:
macro["housing_growth_yoy"] = (
    macro["housing_starts"].pct_change(12) * 100
)

In [32]:
macro["unemployment_change_3m"] = (
    macro["unemployment"] -
    macro["unemployment"].shift(3)
)

In [33]:
macro["claims_change_3m"] = (
    macro["initial_claims"].pct_change(3) * 100
)

In [34]:
future_recession = pd.concat(
    [
        macro["recession"].shift(-1),
        macro["recession"].shift(-2),
        macro["recession"].shift(-3),
        macro["recession"].shift(-4),
        macro["recession"].shift(-5),
        macro["recession"].shift(-6),
    ],
    axis=1
)

macro["recession_next_6m"] = future_recession.max(axis=1)

In [35]:
macro[
    ["date", "recession", "recession_next_6m"]
].head(20)

,date,recession,recession_next_6m
0,1990-01-01,0,0.0
1,1990-02-01,0,1.0
2,1990-03-01,0,1.0
3,1990-04-01,0,1.0
4,1990-05-01,0,1.0
5,1990-06-01,0,1.0
6,1990-07-01,0,1.0
7,1990-08-01,1,1.0
8,1990-09-01,1,1.0
9,1990-10-01,1,1.0


In [36]:
macro["recession_next_6m"].value_counts(dropna=False)

recession_next_6m
0.0    381
1.0     56
NaN      1
Name: count, dtype: int64

In [37]:
macro.loc[
    (macro["date"] >= "2007-01-01") &
    (macro["date"] <= "2008-06-01"),
    ["date", "recession", "recession_next_6m"]
]

,date,recession,recession_next_6m
204,2007-01-01,0,0.0
205,2007-02-01,0,0.0
206,2007-03-01,0,0.0
207,2007-04-01,0,0.0
208,2007-05-01,0,0.0
209,2007-06-01,0,0.0
210,2007-07-01,0,1.0
211,2007-08-01,0,1.0
212,2007-09-01,0,1.0
213,2007-10-01,0,1.0


In [38]:
macro.loc[
    macro.index[-6:],
    "recession_next_6m"
] = np.nan

In [39]:
macro.tail(10)[
    ["date", "recession", "recession_next_6m"]
]

,date,recession,recession_next_6m
428,2025-09-01,0,0.0
429,2025-10-01,0,0.0
430,2025-11-01,0,0.0
431,2025-12-01,0,0.0
432,2026-01-01,0,NaN
433,2026-02-01,0,NaN
434,2026-03-01,0,NaN
435,2026-04-01,0,NaN
436,2026-05-01,0,NaN
437,2026-06-01,0,NaN


In [40]:
features = [
    "unemployment",
    "yield_curve",
    "fed_funds_rate",
    "inflation_yoy",
    "industrial_growth_yoy",
    "housing_growth_yoy",
    "unemployment_change_3m",
    "claims_change_3m",
    "consumer_sentiment"
]

In [41]:
model_data = macro[
    ["date"] + features + ["recession_next_6m"]
].copy()

In [42]:
model_data.head(15)

,date,unemployment,yield_curve,fed_funds_rate,inflation_yoy,industrial_growth_yoy,housing_growth_yoy,unemployment_change_3m,claims_change_3m,consumer_sentiment,recession_next_6m
0,1990-01-01,5.4,0.121429,8.23,NaN,NaN,NaN,NaN,NaN,93.0,0.0
1,1990-02-01,5.3,0.102632,8.24,NaN,NaN,NaN,NaN,NaN,89.5,1.0
2,1990-03-01,5.2,-0.038182,8.28,NaN,NaN,NaN,NaN,NaN,91.3,1.0
3,1990-04-01,5.4,0.061500,8.26,NaN,NaN,NaN,0.0,0.207756,93.9,1.0
4,1990-05-01,5.4,0.115909,8.18,NaN,NaN,NaN,0.1,-0.837404,90.6,1.0
5,1990-06-01,5.2,0.129048,8.29,NaN,NaN,NaN,0.0,4.982619,88.3,1.0
6,1990-07-01,5.5,0.314286,8.15,NaN,NaN,NaN,0.1,1.451279,88.2,1.0
7,1990-08-01,5.7,0.691304,8.13,NaN,NaN,NaN,0.3,8.163265,76.4,1.0
8,1990-09-01,5.9,0.813684,8.20,NaN,NaN,NaN,0.7,8.719647,72.8,1.0
9,1990-10-01,5.9,0.841818,8.11,NaN,NaN,NaN,0.4,15.871935,63.9,1.0


In [43]:
model_data.isna().sum()

date                       0
unemployment               0
yield_curve                0
fed_funds_rate             0
inflation_yoy             12
industrial_growth_yoy     12
housing_growth_yoy        12
unemployment_change_3m     3
claims_change_3m           3
consumer_sentiment         0
recession_next_6m          6
dtype: int64

In [44]:
model_data = model_data.dropna().reset_index(drop=True)

In [45]:
model_data.isna().sum()

date                      0
unemployment              0
yield_curve               0
fed_funds_rate            0
inflation_yoy             0
industrial_growth_yoy     0
housing_growth_yoy        0
unemployment_change_3m    0
claims_change_3m          0
consumer_sentiment        0
recession_next_6m         0
dtype: int64

In [46]:
model_data.to_csv(
    "../data/processed/model_data.csv",
    index=False
)

In [47]:
model_data["recession_next_6m"].value_counts()

recession_next_6m
0.0    375
1.0     45
Name: count, dtype: int64

In [48]:
model_data["recession_next_6m"].value_counts(
    normalize=True
)

recession_next_6m
0.0    0.892857
1.0    0.107143
Name: proportion, dtype: float64

In [49]:
train = model_data[
    model_data["date"] < "2016-01-01"
].copy()

test = model_data[
    model_data["date"] >= "2016-01-01"
].copy()

In [50]:
print(
    "Training:",
    train["date"].min(),
    "to",
    train["date"].max()
)

print(
    "Testing:",
    test["date"].min(),
    "to",
    test["date"].max()
)

Training: 1991-01-01 00:00:00 to 2015-12-01 00:00:00
Testing: 2016-01-01 00:00:00 to 2025-12-01 00:00:00


In [51]:
X_train = train[features]
y_train = train["recession_next_6m"]

X_test = test[features]
y_test = test["recession_next_6m"]

In [52]:
print(X_train.shape)
print(y_train.shape)

print(X_test.shape)
print(y_test.shape)

(300, 9)
(300,)
(120, 9)
(120,)
